In [1]:
import cv2
import mediapipe as mp
import numpy as np

c:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
mp_pose = mp.solutions.pose                            
mp_drawing = mp.solutions.drawing_utils                
pose = mp_pose.Pose()                                  

In [3]:
def get_Angle(point1,point2 ,point3):
    x1,y1 = point1.x ,point1.y
    x2,y2 = point2.x ,point2.y
    x3,y3 = point3.x ,point3.y

    angle = np.degrees(
        np.arctan2(y3 - y2 ,x3 - x2) - 
        np.arctan2(y1 - y2 , x1 - x2)
        )
    
    angle = abs(angle)
    if angle > 180:
        angle = 360 - angle
    return angle

In [10]:
video = cv2.VideoCapture(0)

prev_left_x = None
prev_right_x = None
wave_count = 0
four_timer = 0

while True:
    suc ,img = video.read()
    if suc :
        img1 = cv2.cvtColor(img ,cv2.COLOR_BGR2RGB)
        result = pose.process(img1)


        detected_signal = "NO SIGNAL"


        if result.pose_landmarks:   
            mp_drawing.draw_landmarks(img,result.pose_landmarks,mp_pose.POSE_CONNECTIONS)
            lm = result.pose_landmarks.landmark   

            left_shoulder = lm[mp_pose.PoseLandmark.LEFT_SHOULDER]
            right_shoulder = lm[mp_pose.PoseLandmark.RIGHT_SHOULDER]
            left_elbow = lm[mp_pose.PoseLandmark.LEFT_ELBOW]
            right_elbow = lm[mp_pose.PoseLandmark.RIGHT_ELBOW]
            left_wrist = lm[mp_pose.PoseLandmark.LEFT_WRIST]
            right_wrist = lm[mp_pose.PoseLandmark.RIGHT_WRIST]
            nose = lm[mp_pose.PoseLandmark.NOSE]

            left_arm_angle = get_Angle(left_shoulder,left_elbow,left_wrist)
            right_arm_angle = get_Angle(right_shoulder,right_elbow,right_wrist)

            # Vertical 
            left_wrist_y = left_wrist.y                       
            right_wrist_y = right_wrist.y                     
            left_shoulder_y = left_shoulder.y                 
            right_shoulder_y = right_shoulder.y               
            nose_y = nose.y                                    

            # Horizontal 
            left_wrist_x = left_wrist.x
            right_wrist_x = right_wrist.x
            left_shoulder_x = left_shoulder.x
            right_shoulder_x = right_shoulder.x  

            #wave for FOUR

            if prev_left_x is not None:
                if abs(left_wrist_x - prev_left_x) >  0.05:
                    wave_count += 1

            if prev_right_x is not None:
                if abs(right_wrist_x - prev_right_x) > 0.05:
                    wave_count += 1

            prev_left_x = left_wrist_x
            prev_right_x = right_wrist_x

                                                               

        # SIX - both arm straight up
            if (left_wrist_y < left_shoulder_y - 0.2 and 
                right_wrist_y < right_shoulder_y - 0.2 and 
                left_arm_angle >160 and 
                right_arm_angle > 160):
                detected_signal = "SIX" 

        # OUT: One arm up (like pointing finger)
            elif((left_wrist_y < left_shoulder_y - 0.15 and
                 90 < left_arm_angle < 150) or 
                 (right_wrist_y < right_shoulder_y - 0.15 and
                 90 < right_arm_angle < 150 )):
                detected_signal = "OUT"

        # WIDE: Both arms straight out horizontally
            elif (abs(left_wrist_y - left_shoulder_y) < 0.1 and
                  abs(right_wrist_y - right_shoulder_y) < 0.1 and
                  left_arm_angle > 150 and 
                  right_arm_angle > 150):
                detected_signal = 'WIDE'

            # NO BALL : one arm - horizontal
            elif ((abs(left_wrist_y - left_shoulder_y) < 0.1 and
                 left_arm_angle > 160) or
                  (abs(right_wrist_y - right_shoulder_y) < 0.1 and
                 right_arm_angle  > 160)):
                detected_signal = "NO BALL"

            # FOUR: One arm waving across chest
            elif wave_count > 4 and (
                abs(left_wrist_y - left_shoulder_y) < 0.2 or
                abs(right_wrist_y - right_shoulder_y) < 0.2):
            
                four_timer = 30   # show for 1 second
                wave_count = 0

            # SHORT RUN : hand touching opposite shoulder
            elif (
                abs(right_wrist_x - right_shoulder_x) < 0.12 and
                abs(right_wrist_y - right_shoulder_y) < 0.12 and
                right_arm_angle < 110
            ):
                detected_signal = "SHORT RUN"



            # DEAD BALL: Arms crossed
            elif (abs(left_wrist_x - right_wrist_x)< 0.09 and 
                   left_wrist_y > left_shoulder_y - 0.2 and 
                   right_wrist_y >right_shoulder_y - 0.2):
                detected_signal = "DEAD BALL"
            else:
                detected_signal = "NO SIGNAL"

            # show FOUR for 1 second
        if four_timer > 0:
            detected_signal = "FOUR"
            four_timer -= 1
                
        if detected_signal != 'NO SIGNAL':
            if detected_signal == "SIX":
                color = (0,255,255)
            elif detected_signal == "OUT":
                color = (0,0,255)
            elif detected_signal == "WIDE":
                color = (255 ,0,0)
            elif detected_signal == "NO BALL":
                color = (0,255,0)
            elif detected_signal == "FOUR":
                color = (255, 0, 255) 
            elif detected_signal == "SHORT RUN":
                color = (0,165,255)   # orange
            else:
                color =(255,255,255)

            cv2.putText(img , f"SIGNAL : {detected_signal}", (30,50),cv2.FONT_HERSHEY_COMPLEX,1,color,2)
        else:
            cv2.putText(img,"No Signal Detected",(30,50),cv2.FONT_HERSHEY_COMPLEX,1,(0,0,0),2)
            
    else:
        break
    
    cv2.imshow("Frame",img)
    if cv2.waitKey(30) & 0XFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()
            
